### Setup

In [2]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch
import json
import logging
from PIL import Image
import transformers
from transformers import CLIPProcessor, CLIPModel, pipeline
from diffusers import StableDiffusionImg2ImgPipeline

# CONFIG
transformers.logging.set_verbosity_error()
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
## DATA
SEED_IMAGE_FOLDER = config.get("SEED_IMAGE_FOLDER")
SEED_LABEL_FOLDER = config.get("SEED_LABEL_FOLDER")
AUGEMNT_IMAGE_FOLDER = config.get("AUGEMNT_KFASHION_IMAGE_FOLDER") # change
AUGEMNT_LABEL_FOLDER = config.get("AUGEMNT_LABEL_CLIP_FOLDER") # change
COMBINED_LABEL_FOLDER = config.get("COMBINED_LABEL_CLIP_EXPAND_FOLDER") # change
## MODEL
device = "cuda:0" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
sd_model = StableDiffusionImg2ImgPipeline.from_pretrained("SG161222/Realistic_Vision_V4.0_noVAE", torch_dtype=torch.float16).to(device)
llm_model = "facebook/bart-large-cnn"

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

### Augment Seed Image with Realism Stable Diffusion

In [ ]:
input_dir = config.get("SEED_IMAGE_FOLDER")
output_dir = config.get("AUGEMNT_IMAGE_FOLDER")
prompt = "Generate a high-quality full-body image of a person wearing a different outfit, while preserving the style, composition, lighting, and background of the original image. Exclude the face. The new outfit should clearly differ from the one in the original photo."

for img_name in os.listdir(input_dir):
    image_path = os.path.join(input_dir, img_name)
    init_image = Image.open(image_path).convert("RGB").resize((800, 800))  

    for i in range(10): # one image per 10 augment 
        result = sd_model(prompt=prompt,
                      image=init_image,
                      strength=0.8,  # 0~1
                      guidance_scale=8, # 7.5~12.5
                      num_inference_steps=100).images[0] # 10~50

        out_path = os.path.join(output_dir, f"{img_name[:-4]}_{i+1}.jpg")
        result.save(out_path)

### Augment Instruction Text with CLIP

In [ ]:
# Define attribute categories
categories = {
    "style": [
        "Street", "Modern", "Classic", "Feminine", "Casual", "Sporty", "Vintage", "Elegant", "Minimal",
        "Bohemian", "Chic", "Preppy", "Grunge", "Punk", "Goth", "Romantic", "Resort", "Business"
    ],
    "color": [
        "Black", "White", "Red", "Blue", "Green", "Yellow", "Pink", "Gray", "Beige",
        "Brown", "Orange", "Purple", "Ivory", "Navy", "Khaki", "Mint", "Lavender", "Mustard", "Burgundy"
    ],
    "pattern": [
        "Solid", "Striped", "Checkered", "Floral", "Leopard", "Abstract",
        "Polka Dot", "Paisley", "Camouflage", "Houndstooth", "Animal Print", "Tie-Dye", "Geometric", "Embroidery"
    ],
    "occasion": [
        "Casual Brunch", "Formal Event", "Park Picnic", "Office Meeting",
        "Wedding Guest", "Date Night", "Beach Vacation", "Travel", "Daily Wear", "Birthday Party", "Business Casual"
    ],
    "season": [
        "Spring", "Summer", "Autumn", "Winter"
    ],
    "clothing": [
        "Outerwear", "Top", "Bottoms", "Dress"
    ],
    "length": [
        "Mini", "Midi", "Maxi",
        "Knee Length", "Ankle Length", "Cropped", "Hip Length", "Floor Length"
    ],
    "collar": [
        "Shirt Collar", "V-neck", "Round neck",
        "Turtleneck", "Collarless", "Boat Neck", "Square Neck", "Halter Neck", "Off-shoulder"
    ],
    "sleeve": [
        "Sleeveless", "Short Sleeve", "3/4 Sleeve", "Long Sleeve",
        "Balloon Sleeve", "Cap Sleeve", "Puff Sleeve", "Bishop Sleeve", "Dolman Sleeve"
    ],
    "fit": [
        "Loose", "Fitted", "Oversized",
        "Slim Fit", "Regular Fit", "Relaxed Fit", "Boxy", "Bodycon"
    ]
}

def generate_instruction_content(attributes):
    clothing_item = attributes['clothing']

    clothing_details = {
        "Length": attributes['length'],
        "Color": attributes['color'],
        "Category": clothing_item,
        "Collar": attributes['collar'],
        "Sleeve Length": attributes['sleeve'],
        "Print": attributes['pattern'],
        "Fit": attributes['fit']
    }

    input_caption = (
        f"Style: {attributes['style']}, "
        f"Outerwear: {clothing_details if clothing_item == 'Outerwear' else '{}'}, "
        f"Bottoms: {clothing_details if clothing_item == 'Bottoms' else '{}'}, "
        f"Dress: {clothing_details if clothing_item == 'Dress' else '{}'}, "
        f"Top: {clothing_details if clothing_item == 'Top' else '{}'}"
    )

    add_info = (
        f"Suitable occasion: {attributes['occasion']}, "
        f"Suitable season: {attributes['season']}, "
        f"Pattern: {attributes['pattern']}, "
        f"Shoes: Unknown, Accessories: None"
    )

    return input_caption, add_info

def process_image_and_generate_caption(image_path):
    image = Image.open(image_path).convert("RGB")
    attributes = {}
    for category, options in categories.items():
        inputs = clip_processor(text=options, images=image, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)
        topk_probs, topk_indices = probs.topk(3, dim=1) # Get top 3 options
        selected = [options[i] for i in topk_indices[0]]
        attributes[category] = selected
    input_caption, add_info = generate_instruction_content(attributes)
    return input_caption, add_info

In [ ]:
for file_name in os.listdir(AUGEMNT_IMAGE_FOLDER):
    image_path = os.path.join(AUGEMNT_IMAGE_FOLDER, file_name)
    print(f"Processing image: {image_path}")
        
    caption, add_info = process_image_and_generate_caption(image_path)
    generated_instruction = {
        "Prompt": "Create a full-body photo of a 20-30s Korean woman in the specified outfit, excluding the face.",
        "Input": {"caption": caption},
        "Add_Info": add_info,
        "Output": file_name,
    }

    save_file_name = f"{file_name.rsplit('.', 1)[0]}.json"
    save_path = os.path.join(AUGEMNT_LABEL_FOLDER, save_file_name)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(generated_instruction, f, ensure_ascii=False, indent=4)
    print(f"{save_file_name} saved in {AUGEMNT_LABEL_FOLDER}.")


### Filter and Expand Instruction Text with LLM

In [3]:
# function for expand text
def expand_text(text, summarizer, max_token_len=128, min_token_len=30):
    result = summarizer(text, max_length=max_token_len, min_length=min_token_len, do_sample=False)
    return result[0]['summary_text']
expand_pipe = pipeline("summarization", model=llm_model, device=device)

def calculate_clip_score(image_path, text, clip_model, clip_processor):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(text=[text], images=image, truncation=True, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()} 
    with torch.no_grad():
        outputs = clip_model(**inputs)
        similarity = torch.nn.functional.cosine_similarity(outputs.image_embeds, outputs.text_embeds)
    return similarity.item()

def expand_filter_text(image_path, text_path, save_text_path):
    for filename in sorted(os.listdir(text_path)):
        file_path = os.path.join(text_path, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Combine text
        combined_text = f"Prompt: {data['Prompt']}\nInput: {data['Input']['caption']}\nAdd_Info: {data['Add_Info']}\n"
        remove_chars = ["'Prompt", "Input", "caption", "'", "{", "}", "[", "]", "\n"]
        for char in remove_chars:
            combined_text = combined_text.replace(char, "")

        if not combined_text.strip():
            print(f"No text : {filename}.")
            continue

        image_file = os.path.splitext(filename)[0] + ".jpg"
        image_file_path = os.path.join(image_path, image_file)
        if not os.path.exists(image_file_path):
            print(f"No image file for {filename}")
            continue

        retry_count = 0
        best_score = -1.0
        best_expand = None

        current_text = expand_text(combined_text, expand_pipe)
        if not isinstance(current_text, str):
            current_text = str(current_text)

        # Repeat : Max 10 times
        while retry_count < 10:
            clip_score = calculate_clip_score(image_file_path, current_text, clip_model, clip_processor)
            print(f"[{filename}] Try #{retry_count+1}: CLIP score = {clip_score:.3f}")

            if clip_score > best_score:
                best_score = clip_score
                best_expand = current_text

            # Next repeat if score is low
            current_text = expand_text(current_text, expand_pipe)
            if not isinstance(current_text, str):
                current_text = str(current_text)
            retry_count += 1

        # Save Best score
        print(f"[{filename}] Best CLIP score = {best_score:.3f}, saving result.")
        output_path = os.path.join(save_text_path, filename)
        with open(output_path, 'w', encoding='utf-8') as out_f:
            json.dump(best_expand, out_f, ensure_ascii=False, indent=2)

- Exapnd Seed Data

In [ ]:
expand_filter_text(SEED_IMAGE_FOLDER, SEED_LABEL_FOLDER, COMBINED_LABEL_FOLDER)

- Expand Augment Data

In [ ]:
expand_filter_text(AUGEMNT_IMAGE_FOLDER, AUGEMNT_LABEL_FOLDER, COMBINED_LABEL_FOLDER)